# ⚔️ LoRA vs QLoRA: เปรียบเทียบแบบ Side-by-Side 🟢

**ระดับ:** เริ่มต้น (ต่อจาก Step 1) | **GPU:** T4 16GB | **เวลา:** ~45 นาที

## ทำความเข้าใจก่อนรัน

```
Full Fine-tune  →  เทรนทุก parameter (8B params) → ต้องการ ~80GB VRAM ❌
LoRA            →  เทรนเฉพาะ adapter             → ต้องการ ~16GB VRAM ✅
QLoRA           →  LoRA + 4-bit quantized base    → ต้องการ ~6GB VRAM  ✅✅
```

| | LoRA | QLoRA |
|--|------|-------|
| Base model | 16-bit (bf16) | 4-bit (NF4) |
| Adapter | 16-bit | 16-bit |
| VRAM (8B) | ~14-16 GB | ~5-6 GB |
| Speed | เร็วกว่า ~20% | ช้ากว่าเล็กน้อย |
| Quality | ดีกว่าเล็กน้อย | ดีมาก (ใกล้เคียง) |
| Colab Free T4 | อาจ OOM | รันได้สบาย ✅ |

---
> ⚡ **เปิด GPU ก่อน!** Runtime → Change runtime type → T4 GPU

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps trl peft accelerate bitsandbytes -q
!pip install datasets -q
print('✅ ติดตั้งเสร็จสิ้น')

## 📦 Imports และ Config ร่วม

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
import torch, time, gc

MAX_SEQ_LENGTH = 2048
MAX_STEPS = 50   # ใช้ 50 steps เพื่อให้เปรียบเทียบได้เร็ว
LORA_R = 16
LORA_ALPHA = 32

ALPACA_PROMPT = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def format_prompts(examples, eos_token):
    texts = []
    for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        texts.append(ALPACA_PROMPT.format(inst, inp, out) + eos_token)
    return {"text": texts}

def get_gpu_memory_gb():
    return round(torch.cuda.max_memory_reserved() / 1024**3, 2)

def reset_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

def run_training(model, tokenizer, dataset, output_dir, label):
    """รัน SFTTrainer และวัดผล VRAM + เวลา"""
    reset_gpu_memory()
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=dataset,
        dataset_text_field="text", max_seq_length=MAX_SEQ_LENGTH, dataset_num_proc=2,
        args=TrainingArguments(
            per_device_train_batch_size=2, gradient_accumulation_steps=4,
            warmup_steps=5, max_steps=MAX_STEPS, learning_rate=2e-4,
            fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
            optim="adamw_8bit", logging_steps=10, output_dir=output_dir,
            seed=42, report_to="none",
        ),
    )
    start = time.time()
    stats = trainer.train()
    return {
        "label": label,
        "time_sec": round(time.time() - start, 1),
        "vram_peak_gb": get_gpu_memory_gb(),
        "train_loss": round(stats.metrics.get("train_loss", 0), 4),
        "samples_per_sec": round(stats.metrics.get("train_samples_per_second", 0), 2),
    }

# โหลด dataset ร่วม (500 ตัวอย่างเพื่อความเร็ว)
print('📥 โหลด Dataset...')
dataset = load_dataset("yahma/alpaca-cleaned", split="train[:500]")
print(f'✅ โหลดสำเร็จ: {len(dataset)} ตัวอย่าง')

## 🅰️ Experiment A: QLoRA (แนะนำสำหรับ Colab Free T4)

**วิธีการ:** โหลด base model แบบ **4-bit NF4** + LoRA adapter 16-bit

```python
load_in_4bit=True  # ← นี่คือ QLoRA
```

Unsloth จัดการ NF4 + double quantization ให้อัตโนมัติ ประหยัด VRAM เพิ่มอีก ~0.4 bits/param

In [ ]:
print('=' * 60)
print('🅰️  EXPERIMENT A: QLoRA (4-bit base model)')
print('=' * 60)

# โหลดโมเดลแบบ 4-bit → นี่คือ QLoRA
qlora_model, qlora_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,   # ← KEY: True = QLoRA
)

qlora_model = FastLanguageModel.get_peft_model(
    qlora_model, r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules="all-linear", lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)

print('📊 QLoRA trainable parameters:')
qlora_model.print_trainable_parameters()

qlora_dataset = dataset.map(
    lambda ex: format_prompts(ex, qlora_tokenizer.eos_token), batched=True
)

qlora_results = run_training(
    qlora_model, qlora_tokenizer, qlora_dataset,
    output_dir="outputs/qlora_model", label="QLoRA (4-bit)"
)

qlora_model.save_pretrained("outputs/qlora_model")
qlora_tokenizer.save_pretrained("outputs/qlora_model")
print(f'✅ QLoRA เสร็จ | เวลา: {qlora_results["time_sec"]}s | VRAM peak: {qlora_results["vram_peak_gb"]} GB')

del qlora_model, qlora_tokenizer, qlora_dataset
reset_gpu_memory()

## 🅱️ Experiment B: LoRA (16-bit base model)

**วิธีการ:** โหลด base model แบบ **16-bit เต็มๆ** + LoRA adapter 16-bit

```python
load_in_4bit=False  # ← นี่คือ LoRA ปกติ
```

> ⚠️ ต้องการ VRAM ~14-16GB | **ถ้า OOM ให้ข้ามไป Experiment C ได้เลย**

In [ ]:
print('=' * 60)
print('🅱️  EXPERIMENT B: LoRA (16-bit base model)')
print('=' * 60)
print('⚠️  ต้องการ VRAM ~14-16GB | ถ้า OOM ให้ข้ามไป Experiment C')

# โหลดโมเดลแบบ 16-bit → นี่คือ LoRA ปกติ
lora_model, lora_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b",  # ← ไม่มี -bnb-4bit = โหลด 16-bit
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=False,               # ← KEY: False = LoRA ปกติ
)

lora_model = FastLanguageModel.get_peft_model(
    lora_model, r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules="all-linear", lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)

print('📊 LoRA trainable parameters:')
lora_model.print_trainable_parameters()

lora_dataset = dataset.map(
    lambda ex: format_prompts(ex, lora_tokenizer.eos_token), batched=True
)

lora_results = run_training(
    lora_model, lora_tokenizer, lora_dataset,
    output_dir="outputs/lora_model", label="LoRA (16-bit)"
)

lora_model.save_pretrained("outputs/lora_model")
lora_tokenizer.save_pretrained("outputs/lora_model")
print(f'✅ LoRA เสร็จ | เวลา: {lora_results["time_sec"]}s | VRAM peak: {lora_results["vram_peak_gb"]} GB')

del lora_model, lora_tokenizer, lora_dataset
reset_gpu_memory()

## 🅲 Experiment C: QLoRA Conservative (สำหรับ VRAM น้อย)

ถ้า T4 ยัง OOM ใน Experiment A ให้ใช้ config นี้:
- `r=8` (ลด trainable params ลงครึ่งหนึ่ง)
- `target_modules` เฉพาะ attention layers (ไม่รวม MLP)
- `batch_size=1` + `gradient_accumulation=8`

เหมาะสำหรับ GPU 6-8GB (RTX 3060, GTX 1080 Ti)

In [ ]:
print('=' * 60)
print('🅲  EXPERIMENT C: QLoRA Conservative (VRAM น้อย)')
print('=' * 60)

conservative_model, conservative_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True,
)

conservative_model = FastLanguageModel.get_peft_model(
    conservative_model,
    r=8,                                              # ← ลดจาก 16 เหลือ 8
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # ← เฉพาะ attention
    lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)

print('📊 QLoRA Conservative trainable parameters:')
conservative_model.print_trainable_parameters()

conservative_dataset = dataset.map(
    lambda ex: format_prompts(ex, conservative_tokenizer.eos_token), batched=True
)

reset_gpu_memory()
conservative_trainer = SFTTrainer(
    model=conservative_model, tokenizer=conservative_tokenizer,
    train_dataset=conservative_dataset, dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH, dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=1,    # ← ลดเหลือ 1
        gradient_accumulation_steps=8,    # ← เพิ่มเป็น 8
        warmup_steps=5, max_steps=MAX_STEPS, learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit", logging_steps=10,
        output_dir="outputs/qlora_conservative", seed=42, report_to="none",
    ),
)

start = time.time()
c_stats = conservative_trainer.train()
conservative_results = {
    "label": "QLoRA Conservative (r=8, attention only)",
    "time_sec": round(time.time() - start, 1),
    "vram_peak_gb": get_gpu_memory_gb(),
    "train_loss": round(c_stats.metrics.get("train_loss", 0), 4),
    "samples_per_sec": round(c_stats.metrics.get("train_samples_per_second", 0), 2),
}

conservative_model.save_pretrained("outputs/qlora_conservative")
conservative_tokenizer.save_pretrained("outputs/qlora_conservative")
print(f'✅ Conservative เสร็จ | เวลา: {conservative_results["time_sec"]}s | VRAM: {conservative_results["vram_peak_gb"]} GB')

del conservative_model, conservative_tokenizer, conservative_dataset
reset_gpu_memory()

## 📊 ตารางเปรียบเทียบผลลัพธ์

In [ ]:
all_results = [qlora_results, conservative_results]
try:
    all_results.insert(1, lora_results)
except NameError:
    pass  # ข้าม LoRA ถ้า OOM

print(f'\n{"Method":<42} {"VRAM (GB)":<12} {"Time (s)":<12} {"Loss":<10} {"Samples/s"}')
print('─' * 85)
for r in all_results:
    print(f'{r["label"]:<42} {r["vram_peak_gb"]:<12} {r["time_sec"]:<12} {r["train_loss"]:<10} {r["samples_per_sec"]}')
print('─' * 85)
print('''
📝 อ่านผลลัพธ์:
  VRAM (GB)   → ยิ่งน้อยยิ่งดี
  Time (s)    → ยิ่งน้อยยิ่งดี
  Loss        → ยิ่งน้อยยิ่งดี (โมเดลเรียนรู้ได้ดีกว่า)
  Samples/s   → ยิ่งมากยิ่งดี (throughput)

💡 ข้อสังเกต:
  - QLoRA ใช้ VRAM น้อยกว่า LoRA ประมาณ 2-3x
  - Train loss ของทั้งคู่ใกล้เคียงกัน
  - สำหรับ Colab Free T4: QLoRA คือตัวเลือกที่ดีที่สุด
''')

## 🧪 เปรียบเทียบคุณภาพคำตอบ (Qualitative)

ถามคำถามเดิมกับทุกโมเดล แล้วดูว่าคุณภาพต่างกันไหม

In [ ]:
import os

TEST_PROMPT = ALPACA_PROMPT.format(
    "อธิบายความแตกต่างระหว่าง list และ tuple ใน Python", "", ""
)

def generate_response(model_path, prompt, label):
    print(f'\n📤 {label}:')
    print('─' * 50)
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs, max_new_tokens=200, temperature=0.7, do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(response[len(prompt):].strip())
    print('─' * 50)
    del model, tokenizer
    reset_gpu_memory()

generate_response("outputs/qlora_model", TEST_PROMPT, "QLoRA (4-bit base)")
generate_response("outputs/qlora_conservative", TEST_PROMPT, "QLoRA Conservative (r=8)")
if os.path.exists("outputs/lora_model"):
    generate_response("outputs/lora_model", TEST_PROMPT, "LoRA (16-bit base)")

## 📋 สรุปคำแนะนำการเลือกใช้

| สถานการณ์ | เลือกใช้ |
|-----------|----------|
| Colab Free (T4 16GB) | QLoRA (`load_in_4bit=True`) |
| Colab Pro (A100 40GB) | LoRA (`load_in_4bit=False`) |
| Local GPU < 12GB | QLoRA Conservative (r=8) |
| Local GPU ≥ 24GB | LoRA (r=32+) |
| Production / Best quality | LoRA r=64 บน A100 |

> **Rule of thumb:** "ถ้า VRAM พอ → LoRA, ถ้าไม่พอ → QLoRA"  
> คุณภาพต่างกันน้อยมาก แต่ VRAM ต่างกันมาก